# Quick Start: Running Moirai 2.0 on BOOM benchmark

This notebook is adapted from the [GiftEval repository](https://github.com/SalesforceAIResearch/gift-eval/tree/main/notebooks) and shows how to run [Moirai 2.0](https://huggingface.co/Salesforce/moirai-2.0-R-small) on the BOOM benchmark.

Moirai 2.0 uses `Moirai2Forecast` / `Moirai2Module` from `uni2ts.model.moirai2` and outputs quantile forecasts directly (no sampling). Unlike Moirai 1.x, the `num_samples` and `patch_size` arguments are removed.

Make sure you download the BOOM benchmark and set the `BOOM` environment variable correctly before running this notebook.

We will use the `Dataset` class from GiftEval to load the data and run the model.

Download BOOM datasets. Calling `download_boom_benchmark` also sets the `BOOM` environment variable with the correct path, which is needed for running the evals below.

In [ ]:
# Moirai 2.0 requires gluonts 0.14.x — its `Moirai2Forecast.forward`
# returns a single tensor, while gluonts 0.15+ `QuantileForecastGenerator`
# expects `(outputs,), loc, scale`. Gift-Eval pins gluonts~=0.15.1 in its
# pyproject.toml and upgrades it on install, so we force it back here.
# Safe to run even if already pinned correctly.
!pip install --quiet 'gluonts~=0.14.3'

In [ ]:
import os
import json
from dotenv import load_dotenv
from dataset_utils import download_boom_benchmark

boom_path = "ChangeMe"
download_boom_benchmark(boom_path)
load_dotenv()

dataset_properties_map = json.load(open("./boom/boom_properties.json"))
all_datasets = list(dataset_properties_map.keys())
print(len(all_datasets))

In [ ]:
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)

# Instantiate the metrics
metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]),
]

## Moirai 2.0 Predictor

We load the Moirai 2.0 module and construct a `Moirai2Forecast` per dataset with the
appropriate `prediction_length`, `target_dim`, and `past_feat_dynamic_real_dim`.
`context_length` is fixed at 2048. See the [Moirai 2.0 example notebook](https://github.com/SalesforceAIResearch/uni2ts/blob/main/example/moirai2_forecast.ipynb) for details.

In [ ]:
from gluonts.model import evaluate_model
import csv
import os
from gluonts.time_feature import get_seasonality
from gift_eval.data import Dataset
import torch
from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module

torch.set_float32_matmul_precision("high")

model_name = "moirai2_small"
output_dir = f"ChangeMe/{model_name}"
os.makedirs(output_dir, exist_ok=True)

csv_file_path = os.path.join(output_dir, "all_results.csv")

with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(
        [
            "dataset",
            "model",
            "eval_metrics/MSE[mean]",
            "eval_metrics/MSE[0.5]",
            "eval_metrics/MAE[0.5]",
            "eval_metrics/MASE[0.5]",
            "eval_metrics/MAPE[0.5]",
            "eval_metrics/sMAPE[0.5]",
            "eval_metrics/MSIS",
            "eval_metrics/RMSE[mean]",
            "eval_metrics/NRMSE[mean]",
            "eval_metrics/ND[0.5]",
            "eval_metrics/mean_weighted_sum_quantile_loss",
            "domain",
            "num_variates",
            "dataset_size",
        ]
    )

module = Moirai2Module.from_pretrained("Salesforce/moirai-2.0-R-small")

for ds_num, ds_name in enumerate(all_datasets):
    print(f"Processing dataset: {ds_name} ({ds_num + 1} of {len(all_datasets)})")
    dataset_term = dataset_properties_map[ds_name]["term"]
    terms = ["short", "medium", "long"]
    for term in terms:
        if (term == "medium" or term == "long") and dataset_term == "short":
            continue
        ds_freq = dataset_properties_map[ds_name]["frequency"]
        ds_config = f"{ds_name}/{ds_freq}/{term}"

        # Moirai 2.0 supports multivariate natively
        dataset = Dataset(name=ds_name, term=term, to_univariate=False, storage_env_var="BOOM")

        model = Moirai2Forecast(
            module=module,
            prediction_length=dataset.prediction_length,
            context_length=2048,
            target_dim=dataset.target_dim,
            feat_dynamic_real_dim=0,
            past_feat_dynamic_real_dim=dataset.past_feat_dynamic_real_dim,
        )
        predictor = model.create_predictor(batch_size=1)

        season_length = get_seasonality(dataset.freq)
        dataset_size = len(dataset.test_data)
        print(f"Dataset size: {dataset_size}")

        res = evaluate_model(
            predictor,
            test_data=dataset.test_data,
            metrics=metrics,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=season_length,
        )

        with open(csv_file_path, "a", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(
                [
                    ds_config,
                    model_name,
                    res["MSE[mean]"][0],
                    res["MSE[0.5]"][0],
                    res["MAE[0.5]"][0],
                    res["MASE[0.5]"][0],
                    res["MAPE[0.5]"][0],
                    res["sMAPE[0.5]"][0],
                    res["MSIS"][0],
                    res["RMSE[mean]"][0],
                    res["NRMSE[mean]"][0],
                    res["ND[0.5]"][0],
                    res["mean_weighted_sum_quantile_loss"][0],
                    dataset_properties_map[ds_name]["domain"],
                    dataset_properties_map[ds_name]["num_variates"],
                    dataset_size,
                ]
            )

        print(f"Results for {ds_name} have been written to {csv_file_path}")

## Results

Running the above cell will generate a csv file called `all_results.csv` under the `results/moirai2_small` folder containing the results for the Moirai 2.0 model on the BOOM benchmark. The csv file will look like this:

In [ ]:
import pandas as pd
df = pd.read_csv(output_dir + "/all_results.csv")
df